# Anatomical subtissue identity in the GTEx CLAMP latent space

The cross-validation folds, out-of-fold random-forest/SHAP importance, and logistic-regression evaluation behind this notebook are produced by the `subtissue_cv_folds_gtex`, `subtissue_fold_rf_shap_gtex`, and `subtissue_lr_eval_gtex` rules (`scripts/gtex/subtissue_cv_folds.py`, `scripts/gtex/subtissue_fold_rf_shap.py`, `scripts/gtex/subtissue_lr_eval.py`). This notebook asks whether the complete per-sample CLAMP latent-variable space derived from bulk GTEx RNA-seq preserves anatomical identity within broad tissues, using all 578 CLAMP latent variables and the same fixed five-fold donor-grouped splits throughout. It reports the resulting logistic-regression fits, donor-profile permutation tests, and donor bootstraps.

💡 **Environment:** `clamp-analyses`

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyprojroot.here import here

OUTPUT_DIR = here("output/03_model_biology/01_gtex/04_subtissues")

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.labelsize": 10.5,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

def confusion_matrix(confusion, analysis, tissue, representation):
    subset = confusion[
        (confusion["Analysis"] == analysis)
        & (confusion["Tissue"] == tissue)
        & (confusion["Representation"] == representation)
    ].copy()
    order = subset.drop_duplicates("True_SMTSD")[["True_SMTSD", "True_Display"]]
    labels = order["True_SMTSD"].tolist()
    display = order.set_index("True_SMTSD")["True_Display"].to_dict()
    matrix = subset.pivot(index="True_SMTSD", columns="Predicted_SMTSD", values="Row_Proportion")
    matrix = matrix.reindex(index=labels, columns=labels).fillna(0.0)
    matrix.index = [display[label] for label in labels]
    matrix.columns = [display[label] for label in labels]
    return matrix

def draw_confusion(matrix, ax, title, cbar=False, cbar_ax=None):
    n_classes = len(matrix)
    annotations = np.full(matrix.shape, "", dtype=object)
    for row in range(n_classes):
        for column in range(n_classes):
            value = matrix.iloc[row, column]
            if n_classes <= 3 or row == column or value >= 0.10:
                annotations[row, column] = f"{value:.0%}"
    sns.heatmap(
        matrix, annot=annotations, fmt="", cmap="YlGn", vmin=0, vmax=1,
        linewidths=0.35, linecolor="white", square=True, ax=ax,
        cbar=cbar, cbar_ax=cbar_ax, annot_kws={"fontsize": 6.4},
    )
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel("Predicted region")
    ax.set_ylabel("True region")
    ax.tick_params(axis="x", rotation=52, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

## Results and validation

In [ ]:
anatomical = pd.read_csv(OUTPUT_DIR / "anatomical_subtissue_results.tsv", sep="\t")
all_smtsd = pd.read_csv(OUTPUT_DIR / "all_smtsd_results.tsv", sep="\t")
supplementary_table = pd.read_csv(OUTPUT_DIR / "subtissue_supplementary_table.tsv", sep="\t")
confusion = pd.read_csv(OUTPUT_DIR / "subtissue_confusion_matrices.tsv", sep="\t").query("Representation == 'FullLV'").copy()
counts = pd.read_csv(OUTPUT_DIR / "subtissue_counts.tsv", sep="\t")
exclusions = pd.read_csv(OUTPUT_DIR / "subtissue_exclusions.tsv", sep="\t")
fold_audit = pd.read_csv(OUTPUT_DIR / "fold_smtsd_audit.tsv", sep="\t")

expected_tissues = {"Adipose Tissue", "Blood Vessel", "Brain", "Colon", "Esophagus", "Heart", "Skin"}
assert set(anatomical["Tissue"]) == expected_tissues
assert len(anatomical) == 7 and len(all_smtsd) == 8
assert anatomical.loc[anatomical["Tissue"] == "Brain", "N_Subtissue_Classes"].item() == 13
assert not counts.query("Included_Anatomical")["SMTSD"].eq("Cells - Cultured fibroblasts").any()
assert (fold_audit.query("Included_All_SMTSD")["N_Samples"] > 0).all()
assert np.allclose(
    anatomical["FullLV_Adjusted_Balanced_Accuracy"],
    (anatomical["FullLV_Balanced_Accuracy"] - 1 / anatomical["N_Subtissue_Classes"])
    / (1 - 1 / anatomical["N_Subtissue_Classes"]),
)

display(anatomical[[
    "Tissue_Display", "N_Samples", "N_Donors",
    "FullLV_Balanced_Accuracy", "FullLV_Macro_F1",
]])


## Subtissue confusion matrices

Donor-grouped out-of-fold confusion matrices for the full CLAMP latent space, one per anatomical tissue. Each cell is the fraction of a true subtissue's samples predicted into each subtissue (row-normalized).

In [ ]:
# Full-latent-space anatomical confusion matrices.
confusion_order = ["Adipose Tissue", "Blood Vessel", "Colon", "Esophagus", "Heart", "Skin"]
fig = plt.figure(figsize=(13, 20))
grid = fig.add_gridspec(4, 2, height_ratios=[2.25, 1, 1, 1], hspace=0.95, wspace=0.48)
brain_result = anatomical.set_index("Tissue").loc["Brain"]
brain_ax = fig.add_subplot(grid[0, :])
draw_confusion(
    confusion_matrix(confusion, "anatomical", "Brain", "FullLV"), brain_ax,
    f"{brain_result['Tissue_Display']} — full CLAMP latent space\n"
    f"BA = {brain_result['FullLV_Balanced_Accuracy']:.3f}; "
    f"macro-F1 = {brain_result['FullLV_Macro_F1']:.3f}",
    cbar=False,
)
for panel_index, tissue in enumerate(confusion_order):
    matrix = confusion_matrix(confusion, "anatomical", tissue, "FullLV")
    result = anatomical.set_index("Tissue").loc[tissue]
    ax = fig.add_subplot(grid[1 + panel_index // 2, panel_index % 2])
    draw_confusion(
        matrix, ax,
        f"{result['Tissue_Display']} — full CLAMP latent space\n"
        f"BA = {result['FullLV_Balanced_Accuracy']:.3f}; "
        f"macro-F1 = {result['FullLV_Macro_F1']:.3f}",
        cbar=False,
    )
fig.subplots_adjust(left=0.10, right=0.90, bottom=0.05, top=0.98)
colorbar_ax = fig.add_axes([0.93, 0.37, 0.012, 0.26])
normalizer = mpl.colors.Normalize(vmin=0, vmax=1)
colorbar = fig.colorbar(mpl.cm.ScalarMappable(norm=normalizer, cmap="YlGn"), cax=colorbar_ax)
colorbar.set_label("Fraction of true subtissue")
plt.show()

## Results summary

In [ ]:
summary_columns = [
    "Tissue_Display", "N_Subtissue_Classes", "N_Samples", "N_Donors",
    "FullLV_Balanced_Accuracy", "FullLV_Adjusted_Balanced_Accuracy",
    "FullLV_Macro_F1", "FullLV_BA_CI_Lower", "FullLV_BA_CI_Upper",
    "FullLV_Permutation_P_Value",
]
display(anatomical[summary_columns])
display(exclusions)
